In [1]:
import os, shutil

proj_src = "/kaggle/input/datasets/rachana240906/hci-project-code/HCI-PROJECT"
assert os.path.exists(proj_src), "Path not found, rerun the diagnostic cell"

dst = "/kaggle/working/HCI-PROJECT"
if os.path.exists(dst):
    shutil.rmtree(dst)
shutil.copytree(proj_src, dst)
print(os.listdir(dst))

['requirements.txt', 'configs', 'docs', 'checklist.md', 'src', 'README.md']


In [2]:
import os
print("Contents of /kaggle/input:", os.listdir("/kaggle/input") if os.path.exists("/kaggle/input") else "MISSING")

for root, dirs, files in os.walk("/kaggle/input"):
    depth = root.count(os.sep) - "/kaggle/input".count(os.sep)
    if depth <= 3:
        print(root, "->", dirs[:10], files[:10])

Contents of /kaggle/input: ['datasets']
/kaggle/input -> ['datasets'] []
/kaggle/input/datasets -> ['rachana240906'] []
/kaggle/input/datasets/rachana240906 -> ['hci-project-code'] []
/kaggle/input/datasets/rachana240906/hci-project-code -> ['HCI-PROJECT'] []


In [3]:
!pip install -q mne mne-bids openneuro-py pyyaml

In [4]:
import mne, mne_bids, openneuro, yaml
print("mne:", mne.__version__)
print("mne_bids:", mne_bids.__version__)
print("openneuro-py: imported OK")

mne: 1.13.2
mne_bids: 0.20.0
openneuro-py: imported OK


In [5]:
import sys
sys.path.insert(0, "/kaggle/working/HCI-PROJECT")

from src.utils import load_config, set_seed
cfg = load_config()
set_seed(cfg["seed"])
print(cfg)

{'project': 'CD-B2M', 'seed': 42, 'seeds': [0, 1, 2, 3, 4], 'dataset': {'name': 'ds004752', 'openneuro_id': 'ds004752'}, 'paths': {'data_root': '/kaggle/working/data/ds004752', 'cache_dir': '/kaggle/working/cache', 'results_dir': '/kaggle/working/results'}, 'conditions': ['real', 'maxent', 'shuffled', 'random'], 'architectures': ['rnn', 'transformer', 'rl'], 'regions': ['hippocampus', 'ofc', 'entorhinal', 'cingulate']}


In [6]:
import urllib.request
print(urllib.request.urlopen("https://openneuro.org", timeout=10).status)

200


In [7]:
import os
import openneuro as on

DATA_DIR = cfg["paths"]["data_root"]
os.makedirs(DATA_DIR, exist_ok=True)

on.download(
    dataset="ds004752",
    target_dir=DATA_DIR,
    include=["README*", "CHANGES", "dataset_description.json",
             "participants.*", "*.tsv", "*.json"],
)

🌍 Preparing to download ds004752 …
📥 Checking 768 files, downloading as needed (5 concurrent downloads). 


Output()

✅ Finished downloading ds004752 (downloaded 768 files and 1.4 MB).


In [8]:
import subprocess
print(subprocess.run(f"ls {DATA_DIR}; echo ---; find {DATA_DIR} -maxdepth 3 -type d | head -60",
                     shell=True, capture_output=True, text=True).stdout)

CHANGES
dataset_description.json
participants.json
participants.tsv
README
sub-01
sub-02
sub-03
sub-04
sub-05
sub-06
sub-07
sub-08
sub-09
sub-10
sub-11
sub-12
sub-13
sub-14
sub-15
---
/kaggle/working/data/ds004752
/kaggle/working/data/ds004752/sub-03
/kaggle/working/data/ds004752/sub-03/ses-03
/kaggle/working/data/ds004752/sub-03/ses-03/ieeg
/kaggle/working/data/ds004752/sub-03/ses-03/eeg
/kaggle/working/data/ds004752/sub-03/ses-02
/kaggle/working/data/ds004752/sub-03/ses-02/ieeg
/kaggle/working/data/ds004752/sub-03/ses-02/eeg
/kaggle/working/data/ds004752/sub-03/ses-01
/kaggle/working/data/ds004752/sub-03/ses-01/ieeg
/kaggle/working/data/ds004752/sub-03/ses-01/eeg
/kaggle/working/data/ds004752/sub-04
/kaggle/working/data/ds004752/sub-04/ses-02
/kaggle/working/data/ds004752/sub-04/ses-02/ieeg
/kaggle/working/data/ds004752/sub-04/ses-02/eeg
/kaggle/working/data/ds004752/sub-04/ses-01
/kaggle/working/data/ds004752/sub-04/ses-01/ieeg
/kaggle/working/data/ds004752/sub-04/ses-01/eeg
/kaggle

In [9]:
import pandas as pd
print(pd.read_csv(f"{DATA_DIR}/participants.tsv", sep="\t").head(20))

   participant_id  usz_identifier  sciAdv_identifier  eLife_identifier  age  \
0          sub-01              28                1.0                 7   24   
1          sub-02              22                2.0                 9   39   
2          sub-03              19                3.0                12   18   
3          sub-04              30                4.0                 6   28   
4          sub-05              33                5.0                 8   20   
5          sub-06              13                6.0                 5   31   
6          sub-07              23                7.0                11   47   
7          sub-08              29                8.0                13   56   
8          sub-09              16                9.0                10   19   
9          sub-10              42                NaN                 1   46   
10         sub-11              38                NaN                 2   19   
11         sub-12              37                NaN

In [10]:
#Cell A
import pathlib, collections, glob
import pandas as pd

root = pathlib.Path(DATA_DIR)
rows = [{"subject": s.name, "n_sessions": len(list(s.glob("ses-*")))}
        for s in sorted(root.glob("sub-*"))]
sess_df = pd.DataFrame(rows)
print(sess_df.to_string(index=False))
print("Total sessions:", sess_df.n_sessions.sum())

subject  n_sessions
 sub-01           4
 sub-02           7
 sub-03           3
 sub-04           2
 sub-05           3
 sub-06           7
 sub-07           4
 sub-08           4
 sub-09           2
 sub-10           2
 sub-11           6
 sub-12           6
 sub-13           6
 sub-14           8
 sub-15           4
Total sessions: 68


In [11]:
#cell B
cnt = collections.Counter()
for p in root.rglob("*"):
    if p.is_file():
        cnt[(p.parent.name if p.parent.name in ("eeg", "ieeg") else "other",
             p.name.split("_")[-1])] += 1
for k, v in sorted(cnt.items()):
    print(k, v)

print("Top-level folders:", [p.name for p in root.iterdir() if p.is_dir() and not p.name.startswith("sub-")])

('eeg', 'channels.tsv') 68
('eeg', 'eeg.json') 68
('eeg', 'events.json') 68
('eeg', 'events.tsv') 68
('ieeg', 'channels.tsv') 68
('ieeg', 'electrodes.json') 68
('ieeg', 'electrodes.tsv') 68
('ieeg', 'events.json') 68
('ieeg', 'events.tsv') 68
('ieeg', 'ieeg.json') 68
('other', 'CHANGES') 1
('other', 'README') 1
('other', 'description.json') 1
('other', 'participants.json') 1
('other', 'participants.tsv') 1
('other', 'scans.tsv') 68
('other', 'sessions.tsv') 15
Top-level folders: []


In [12]:
#cell C
print(open(root / "README").read()[:3000])

### Dataset of intracranial EEG, scalp EEG and beamforming sources from human epilepsy patients performing a verbal working memory task

#### Description
We present an electrophysiological dataset recorded from fifteen subjects during a verbal working memory task. Subjects were epilepsy patients undergoing intracranial monitoring for localization of epileptic seizures. Subjects performed a modified Sternberg task in which the encoding of memory items, maintenance, and recall were temporally separated. The dataset includes simultaneously recorded scalp EEG with the 10-20 system, intracranial EEG (iEEG) recorded with depth electrodes, waveforms, and the MNI coordinates and anatomical labels of all intracranial electrodes. The dataset includes also reconstructed virtual sensor data that were created by performing LCMV beamforming on the EEG at specific brain regions including, temporal superior lobe, lateral prefrontal cortex, occipital cortex, posterior parietal cortex, and Broca.  Subje

In [13]:
#Cell D
def show(pattern, n=12):
    fs = sorted(glob.glob(str(root / pattern)))
    print("\n###", pattern, "->", len(fs), "files")
    if not fs:
        return None
    print(fs[0])
    df = pd.read_csv(fs[0], sep="\t")
    print("shape:", df.shape)
    print(df.head(n).to_string())
    return df

ev = show("sub-01/ses-01/ieeg/*events.tsv", 15)
ch = show("sub-01/ses-01/ieeg/*channels.tsv")
el = show("sub-01/ses-01/ieeg/*electrodes.tsv")


### sub-01/ses-01/ieeg/*events.tsv -> 1 files
/kaggle/working/data/ds004752/sub-01/ses-01/ieeg/sub-01_ses-01_task-verbalWM_run-01_events.tsv
shape: (50, 11)
       onset  duration  nTrial  begSample  endSample  SetSize ProbeLetter Match  Correct  ResponseTime  Artifact
0     0.0005         8       1          1      16000        8           H   OUT        1       2.48400         0
1     8.0005         8       2      16001      32000        4           T    IN        1       1.66775         0
2    16.0005         8       3      32001      48000        8           L   OUT        1       1.47200         0
3    24.0005         8       4      48001      64000        6           G   OUT        1       1.30875         0
4    32.0005         8       5      64001      80000        8           S   OUT        1       1.51625         0
5    40.0005         8       6      80001      96000        4           R    IN        1       0.89900         0
6    48.0005         8       7      96001     11200

In [14]:
#cell E
import json
for f in sorted(glob.glob(str(root / "sub-01/ses-01/ieeg/*.json")))[:3]:
    print("\n###", f)
    print(json.dumps(json.load(open(f)), indent=1)[:1500])


### /kaggle/working/data/ds004752/sub-01/ses-01/ieeg/sub-01_ses-01_task-verbalWM_run-01_electrodes.json
{
 "name": {
  "Description": "the label of the electrode",
  "Type": "object"
 },
 "x": {
  "Description": "the MNI coordinate of the electrode in the x-axis",
  "Type": "object"
 },
 "y": {
  "Description": "the MNI coordinate of the electrode in the y-axis",
  "Type": "object"
 },
 "z": {
  "Description": "the MNI coordinate of the electrode in the z-axis",
  "Type": "object"
 },
 "size": {
  "Description": "the size of electrode in mm if it is a depth electrode size defines the diameter, if it is an ECOG electrode size defines the contact surface in mm^2",
  "Type": "object",
  "Units": "mm"
 },
 "AnatomicalLocation": {
  "Description": "the anatomical location of the electrode after mapping its MNI coordinates to the Brainnetome atlas",
  "Type": "object"
 }
}

### /kaggle/working/data/ds004752/sub-01/ses-01/ieeg/sub-01_ses-01_task-verbalWM_run-01_events.json
{
 "onset": {
  "D

In [15]:
print(ev.columns.tolist())
for c in ev.columns:
    if ev[c].nunique() < 15:
        print(c, "->", ev[c].value_counts(dropna=False).to_dict())

['onset', 'duration', 'nTrial', 'begSample', 'endSample', 'SetSize', 'ProbeLetter', 'Match', 'Correct', 'ResponseTime', 'Artifact']
duration -> {8: 50}
SetSize -> {4: 21, 8: 17, 6: 12}
Match -> {'OUT': 30, 'IN': 20}
Correct -> {1: 44, 0: 6}
Artifact -> {0: 48, 1: 2}


In [16]:
import pathlib, pandas as pd
DATA_DIR = "/kaggle/working/data/ds004752"
root = pathlib.Path(DATA_DIR)

In [17]:
#cell F
print(open(root / "README").read()[3000:8000])

C; orbitofrontal cortex, PPC; posterior parietal cortex, AC; auditory cortex, V1; primary visual cortex

### BIDS Conversion

bids-starter-kid and custom Matlab scripts were used to convert the dataset into BIDS format.

### References
[1] Dimakopoulos V, Megevand P, Stieglitz LH, Imbach L, Sarnthein J. Information flows from hippocampus to auditory cortex during replay of verbal working memory items. Elife 2022;11. 10.7554/eLife.78677

[2] Boran E, Fedele T, Klaver P, Hilfiker P, Stieglitz L, Grunwald T, et al. Persistent hippocampal neural firing and hippocampal-cortical coupling predict verbal working memory load. Science Advances 2019;5(3):eaav3687. 10.1126/sciadv.aav3687

[3] Boran E, Fedele T, Steiner A, Hilfiker P, Stieglitz L, Grunwald T, et al. Dataset of human medial temporal lobe neurons, scalp and intracranial EEG during a verbal working memory task. Scientific Data 2020;7(1):30. 10.1038/s41597-020-0364-3


In [18]:
#cell G
rows = []
for f in sorted(root.glob("sub-*/ses-*/ieeg/*_electrodes.tsv")):
    d = pd.read_csv(f, sep="\t")
    d["subject"], d["session"] = f.parts[-4], f.parts[-3]
    rows.append(d)
el_all = pd.concat(rows, ignore_index=True)

el_all["gyrus"] = el_all["AnatomicalLocation"].astype(str).str.split(",").str[0].str.strip()
el_all["is_entorhinal"] = el_all["AnatomicalLocation"].astype(str).str.contains("entorhinal|28/34", case=False, regex=True)

# Are electrode sets stable across a subject's sessions?
chk = el_all.groupby("subject").agg(n_unique_electrodes=("name", "nunique"),
                                    n_rows=("name", "size"),
                                    n_sessions=("session", "nunique"))
print(chk, "\n")

# One row per subject-electrode so sessions don't inflate counts
el_u = el_all.drop_duplicates(["subject", "name"])
print("Unique electrodes:", len(el_u))
print("no_label_found fraction:", round((el_u.gyrus == "no_label_found").mean(), 3), "\n")
print(el_u.gyrus.value_counts().to_string(), "\n")

cand = el_u[el_u.gyrus.isin(["Hipp", "OrG", "PhG", "CG", "Amyg"]) | el_u.is_entorhinal].copy()
cand.loc[cand.is_entorhinal, "gyrus"] = "Entorhinal(A28/34)"
print(pd.crosstab(cand.subject, cand.gyrus).to_string())

         n_unique_electrodes  n_rows  n_sessions
subject                                         
sub-01                    48     192           4
sub-02                    64     448           7
sub-03                    40     120           3
sub-04                    64     128           2
sub-05                    32      96           3
sub-06                    64     448           7
sub-07                    64     256           4
sub-08                    62     248           4
sub-09                    64     128           2
sub-10                    80     160           2
sub-11                    36     216           6
sub-12                    46     276           6
sub-13                    68     408           6
sub-14                    64     512           8
sub-15                    64     256           4 

Unique electrodes: 860
no_label_found fraction: 0.301 

gyrus
no_label_found    259
MTG               198
Hipp               97
ITG                64
STG            

In [19]:
#cell H
rows = []
for f in sorted(root.glob("sub-*/ses-*/ieeg/*_events.tsv")):
    d = pd.read_csv(f, sep="\t")
    d["subject"], d["session"] = f.parts[-4], f.parts[-3]
    rows.append(d)
ev_all = pd.concat(rows, ignore_index=True)

print("Total trials:", len(ev_all))
print("Columns identical across files:", len({tuple(pd.read_csv(f, sep='\t', nrows=0).columns)
      for f in root.glob('sub-*/ses-*/ieeg/*_events.tsv')}) == 1)
print("\nSetSize:", ev_all.SetSize.value_counts().to_dict())
print("Match:", ev_all.Match.value_counts().to_dict())
print("Artifact trials:", int(ev_all.Artifact.sum()))
print("Overall accuracy:", round(ev_all.Correct.mean(), 3))
print("\nAccuracy by SetSize:\n", ev_all.groupby("SetSize").Correct.mean().round(3).to_string())
print("\nMedian RT by SetSize:\n", ev_all.groupby("SetSize").ResponseTime.median().round(3).to_string())
print("\nTrials per subject:\n", ev_all.groupby("subject").size().to_string())
print("\nSetSize x subject:\n", pd.crosstab(ev_all.subject, ev_all.SetSize).to_string())

Total trials: 3353
Columns identical across files: True

SetSize: {4: 1281, 8: 1057, 6: 998, 0: 17}
Match: {'OUT': 1714, 'IN': 1622}
Artifact trials: 168
Overall accuracy: 0.908

Accuracy by SetSize:
 SetSize
0    0.000
4    0.977
6    0.903
8    0.845

Median RT by SetSize:
 SetSize
0    0.000
4    1.083
6    1.160
8    1.249

Trials per subject:
 subject
sub-01    200
sub-02    339
sub-03    147
sub-04     99
sub-05    150
sub-06    349
sub-07    197
sub-08    200
sub-09     97
sub-10     77
sub-11    300
sub-12    300
sub-13    300
sub-14    398
sub-15    200

SetSize x subject:
 SetSize   0    4    6    8
subject                   
sub-01    0   72   59   69
sub-02    0  137   97  105
sub-03    0   53   48   46
sub-04    0   37   34   28
sub-05    0   57   49   44
sub-06    0  144   95  110
sub-07    0   68   67   62
sub-08    0   83   50   67
sub-09    0   30   34   33
sub-10    0   24   28   25
sub-11    0  117   80  103
sub-12    1  119   80  100
sub-13    0  100  101   99
sub-1

In [20]:
#cell 1: real filenames
import glob, pandas as pd
for f in sorted(glob.glob(f"{DATA_DIR}/sub-01/**/*scans.tsv", recursive=True))[:2]:
    print(f)
    print(pd.read_csv(f, sep="\t").to_string())

/kaggle/working/data/ds004752/sub-01/ses-01/sub-01_ses-01_scans.tsv
                                         filename             acq_time
0  eeg/sub-01_ses-01_task-verbalWM_run-01_eeg.edf  2016-12-14T17:13:33
/kaggle/working/data/ds004752/sub-01/ses-02/sub-01_ses-02_scans.tsv
                                         filename             acq_time
0  eeg/sub-01_ses-02_task-verbalWM_run-01_eeg.edf  2016-12-14T17:36:24


In [21]:
#cell 2: download one session’s whole iEEG folder. A folder include doesn’t depend on the filename.
import openneuro as on, subprocess
on.download(dataset="ds004752", target_dir=DATA_DIR,
            include=["sub-01/ses-01/ieeg"], max_retries=5)
print(subprocess.run(f"ls -la {DATA_DIR}/sub-01/ses-01/ieeg", shell=True,
                     capture_output=True, text=True).stdout)

🌍 Preparing to download ds004752 …
📥 Checking 12 files, downloading as needed (5 concurrent downloads). 


Output()

✅ Finished downloading ds004752 (downloaded 1 file and 73.3 MB).
total 75052
drwxr-xr-x 2 root root     4096 Oct 25 08:24 .
drwxr-xr-x 4 root root     4096 Oct 25 08:24 ..
-rw-r--r-- 1 root root     1483 Oct 25 08:24 sub-01_ses-01_task-verbalWM_run-01_channels.tsv
-rw-r--r-- 1 root root      915 Oct 25 08:24 sub-01_ses-01_task-verbalWM_run-01_electrodes.json
-rw-r--r-- 1 root root     5073 Oct 25 08:24 sub-01_ses-01_task-verbalWM_run-01_electrodes.tsv
-rw-r--r-- 1 root root     1732 Oct 25 08:24 sub-01_ses-01_task-verbalWM_run-01_events.json
-rw-r--r-- 1 root root     2453 Oct 25 08:24 sub-01_ses-01_task-verbalWM_run-01_events.tsv
-rw-r--r-- 1 root root 76812544 Oct 25 08:24 sub-01_ses-01_task-verbalWM_run-01_ieeg.edf
-rw-r--r-- 1 root root     1366 Oct 25 08:24 sub-01_ses-01_task-verbalWM_run-01_ieeg.json



In [22]:
#cell I
el_u = el_all.drop_duplicates(["subject", "name"]).copy()

for g in ["Hipp", "PhG", "Amyg"]:
    print(f"--- {g} subregions ---")
    print(el_u[el_u.gyrus == g].AnatomicalLocation.value_counts().to_string(), "\n")

groups = {"hippocampus": ["Hipp"], "amygdala": ["Amyg"], "parahippocampal": ["PhG"],
          "lateral_temporal": ["STG", "MTG", "ITG", "pSTS"],
          "unlabeled": ["no_label_found", "nan"]}
g2r = {g: r for r, gs in groups.items() for g in gs}
el_u["region"] = el_u.gyrus.map(g2r).fillna("other_labeled")

print(pd.crosstab(el_u.subject, el_u.region).to_string())
print("\nTotal electrodes per subject:\n", el_u.groupby("subject").size().to_string())

--- Hipp subregions ---
AnatomicalLocation
Hipp, Left Hippocampus cHipp, caudal hippocampus      38
Hipp, Right Hippocampus cHipp, caudal hippocampus     27
Hipp, Left Hippocampus rHipp, rostral hippocampus     22
Hipp, Right Hippocampus rHipp, rostral hippocampus     9
Hipp, Right Hippocampus                                1 

--- PhG subregions ---
AnatomicalLocation
PhG, Right Parahippocampal Gyrus A35/36r, rostral area 35/36                                    11
PhG, Left Parahippocampal Gyrus A35/36r, rostral area 35/36                                     10
PhG, Left Parahippocampal Gyrus TI, area TI (temporal agranular insular cortex)                  3
PhG, Left Parahippocampal Gyrus TL, area TL (lateral PPHC, posterior parahippocampal gyrus)      3
PhG, Right Parahippocampal Gyrus TL, area TL (lateral PPHC, posterior parahippocampal gyrus)     2
PhG, Left Parahippocampal Gyrus A28/34, area 28/34 (EC, entorhinal cortex)                       1
PhG, Right Parahippocampal Gyrus A

In [23]:
#cell J
clean = ev_all[(ev_all.SetSize > 0) & (ev_all.Artifact == 0)]
print("Clean trials:", len(clean), "of", len(ev_all))
print("Accuracy on clean trials:", round(clean.Correct.mean(), 3))
print(pd.crosstab(clean.subject, clean.SetSize).to_string())
print("\nClean trials per subject-session (min/median/max):",
      clean.groupby(["subject", "session"]).size().agg(["min", "median", "max"]).tolist())

Clean trials: 3185 of 3353
Accuracy on clean trials: 0.911
SetSize    4    6    8
subject               
sub-01    61   51   64
sub-02   126   96   97
sub-03    44   46   44
sub-04    36   31   27
sub-05    52   46   35
sub-06   141   93  104
sub-07    66   62   58
sub-08    61   46   59
sub-09    27   29   32
sub-10    24   28   25
sub-11   117   80  103
sub-12   119   80  100
sub-13   100   99   96
sub-14   154  115  111
sub-15    85   60   55

Clean trials per subject-session (min/median/max): [33.0, 48.0, 50.0]


In [24]:
#cell 1
!pip install -q mne mne-bids openneuro-py pyyaml
import os, pathlib, subprocess
import pandas as pd
import openneuro as on

DATA_DIR = "/kaggle/working/data/ds004752"
os.makedirs(DATA_DIR, exist_ok=True)

on.download(dataset="ds004752", target_dir=DATA_DIR,
            include=["README*", "CHANGES", "dataset_description.json",
                     "participants.*", "*.tsv", "*.json"])

🌍 Preparing to download ds004752 …
📥 Checking 768 files, downloading as needed (5 concurrent downloads). 


Output()

✅ Finished downloading ds004752 (downloaded 0 files and 0 B).


In [25]:
root = pathlib.Path(DATA_DIR)
subjects = sorted(p.name for p in root.glob("sub-*"))
print(len(subjects), "subjects;",
      len(list(root.glob("sub-*/ses-*"))), "sessions (expect 15 and 68)")

failed = []
for sub in subjects:
    folders = [f"{sub}/{s.name}/ieeg" for s in sorted((root / sub).glob("ses-*"))]
    try:
        on.download(dataset="ds004752", target_dir=DATA_DIR,
                    include=folders, max_retries=5, max_concurrent_downloads=4)
        print("OK", sub, len(folders), "sessions")
    except Exception as e:
        print("FAILED", sub, repr(e)[:200])
        failed.append(sub)
print("Failed subjects:", failed)

15 subjects; 68 sessions (expect 15 and 68)
🌍 Preparing to download ds004752 …
📥 Checking 33 files, downloading as needed (4 concurrent downloads). 


Output()

✅ Finished downloading ds004752 (downloaded 3 files and 219.8 MB).
OK sub-01 4 sessions
🌍 Preparing to download ds004752 …
📥 Checking 54 files, downloading as needed (4 concurrent downloads). 


Output()

✅ Finished downloading ds004752 (downloaded 7 files and 662.2 MB).
OK sub-02 7 sessions
🌍 Preparing to download ds004752 …
📥 Checking 26 files, downloading as needed (4 concurrent downloads). 


Output()

... (21 outputs left out)

Output()

✅ Finished downloading ds004752 (downloaded 8 files and 1.5 GB).
OK sub-14 8 sessions
🌍 Preparing to download ds004752 …
📥 Checking 33 files, downloading as needed (4 concurrent downloads). 


Output()

✅ Finished downloading ds004752 (downloaded 4 files and 800.1 MB).
OK sub-15 4 sessions
Failed subjects: []


In [26]:
edfs = sorted(root.glob("sub-*/ses-*/ieeg/*_ieeg.edf"))
sizes = [p.stat().st_size for p in edfs]
print("EDF files:", len(edfs), "(expect 68)")
print("Total GB:", round(sum(sizes) / 1e9, 2), "| smallest MB:", round(min(sizes) / 1e6, 1))
print("Tiny files (should be 0):", sum(s < 10_000_000 for s in sizes))

EDF files: 68 (expect 68)
Total GB: 8.84 | smallest MB: 51.2
Tiny files (should be 0): 0


In [27]:
import mne
f = next(root.glob("sub-01/ses-01/ieeg/*_ieeg.edf"))
raw = mne.io.read_raw_edf(f, preload=False, verbose="ERROR")
ch = pd.read_csv(str(f).replace("_ieeg.edf", "_channels.tsv"), sep="\t")
ev = pd.read_csv(str(f).replace("_ieeg.edf", "_events.tsv"), sep="\t")

print("sfreq:", raw.info["sfreq"], "| channels:", len(raw.ch_names),
      "| samples:", raw.n_times, "| seconds:", raw.n_times / raw.info["sfreq"])
print("Channel names match channels.tsv:", raw.ch_names == ch.name.tolist())
print("Last endSample in events:", ev.endSample.max())

sfreq: 2000.0 | channels: 48 | samples: 800000 | seconds: 400.0
Channel names match channels.tsv: True
Last endSample in events: 800000


In [28]:
os.makedirs("/kaggle/working/meta", exist_ok=True)

def stack(pattern):
    rows = []
    for f in sorted(root.glob(pattern)):
        d = pd.read_csv(f, sep="\t")
        d["subject"], d["session"] = f.parts[-4], f.parts[-3]
        rows.append(d)
    return pd.concat(rows, ignore_index=True)

el = stack("sub-*/ses-*/ieeg/*_electrodes.tsv")
el["AnatomicalLocation"] = el["AnatomicalLocation"].astype(str)
el["gyrus"] = el.AnatomicalLocation.str.split(",").str[0].str.strip()
groups = {"hippocampus": ["Hipp"], "amygdala": ["Amyg"], "parahippocampal": ["PhG"],
          "lateral_temporal": ["STG", "MTG", "ITG", "pSTS"],
          "unlabeled": ["no_label_found", "nan"]}
g2r = {g: r for r, gs in groups.items() for g in gs}
el["region"] = el.gyrus.map(g2r).fillna("other_labeled")
el.to_csv("/kaggle/working/meta/electrodes_all.csv", index=False)

ev_all = stack("sub-*/ses-*/ieeg/*_events.tsv")
ev_all["clean"] = ((ev_all.SetSize > 0) & (ev_all.Artifact == 0)).astype(int)
ev_all.to_csv("/kaggle/working/meta/trials_all.csv", index=False)

print(el.drop_duplicates(["subject", "name"]).region.value_counts().to_string())
print("Clean trials:", int(ev_all.clean.sum()), "of", len(ev_all))

region
lateral_temporal    332
unlabeled           262
other_labeled        99
hippocampus          97
amygdala             39
parahippocampal      31
Clean trials: 3185 of 3353
